# Tutoriel pgvector — Recherche vectorielle avec PostgreSQL & Python

 ### Partie 1 — Setup & connexion Python 

In [6]:
import psycopg2
from pgvector.psycopg2 import register_vector
import numpy as np

In [ ]:
# Connexion à PostgreSQL
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    dbname="vectordb",
    user="postgres",
    password="secret"
)

conn.autocommit = True
cur = conn.cursor()

# 1. Activer l'extension AVANT register_vector
cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
print("✅ Extension pgvector activée")

# 2. Maintenant pgvector existe dans PostgreSQL
register_vector(conn)

# Vérifier la version
cur.execute("""
    SELECT extversion
    FROM pg_extension
    WHERE extname = 'vector';
""")

version = cur.fetchone()[0]
print(f"Version pgvector : {version}")

cur.close()
conn.close()

✅ Extension pgvector activée
Version pgvector : 0.8.2


In [ ]:
# ex01_table.py
conn = psycopg2.connect(host="localhost", dbname="vectordb",
                        user="postgres", password="secret")
conn.autocommit = True
register_vector(conn)
cur = conn.cursor()

cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")

# Création d'une table "produits" avec 3 colonnes :
# - id : clé primaire auto-incrémentée
# - nom : le nom du produit
# - caracteristiques : vecteur de dimension 3
#   Chaque dimension représente une propriété :
#   [sucré (0-1), épicé (0-1), exotique (0-1)]
cur.execute("""
    DROP TABLE IF EXISTS produits;
    CREATE TABLE produits (
        id               SERIAL PRIMARY KEY,
        nom              TEXT NOT NULL,
        caracteristiques vector(3)
    );
""")
print("✅ Table 'produits' créée")

# Afficher la structure de la table
cur.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_name = 'produits'
    ORDER BY ordinal_position;
""")
print("\nStructure de la table :")
for col, dtype in cur.fetchall():
    print(f"  {col:25s} {dtype}")

cur.close()
conn.close()

✅ Table 'produits' créée

Structure de la table :
  id                        integer
  nom                       text
  caracteristiques          USER-DEFINED


In [4]:
conn = psycopg2.connect(host="localhost", dbname="vectordb",
                        user="postgres", password="secret")
conn.autocommit = True
register_vector(conn)
cur = conn.cursor()

# Données : [sucré, épicé, exotique]
produits = [
    ("mangue",       np.array([0.9, 0.1, 0.95])),
    ("piment",       np.array([0.1, 0.95, 0.6])),
    ("fraise",       np.array([0.85, 0.05, 0.2])),
    ("curry",        np.array([0.2, 0.9, 0.7])),
    ("vanille",      np.array([0.95, 0.02, 0.3])),
    ("wasabi",       np.array([0.05, 0.98, 0.55])),
    ("ananas",       np.array([0.8, 0.08, 0.88])),
]

# Insertion un par un
for nom, vec in produits:
    cur.execute(
        "INSERT INTO produits (nom, caracteristiques) VALUES (%s, %s)",
        (nom, vec)  # numpy array → pgvector sait le convertir
    )

print(f"✅ {len(produits)} produits insérés")

# Vérification : lire les données insérées
cur.execute("SELECT id, nom, caracteristiques FROM produits ORDER BY id;")
print("\nProduits en base :")
for id_, nom, vec in cur.fetchall():
    print(f"  [{id_}] {nom:12s} → {np.round(vec, 2)}")

cur.close()
conn.close()

✅ 7 produits insérés

Produits en base :
  [1] mangue       → [0.9  0.1  0.95]
  [2] piment       → [0.1  0.95 0.6 ]
  [3] fraise       → [0.85 0.05 0.2 ]
  [4] curry        → [0.2 0.9 0.7]
  [5] vanille      → [0.95 0.02 0.3 ]
  [6] wasabi       → [0.05 0.98 0.55]
  [7] ananas       → [0.8  0.08 0.88]


In [5]:
conn = psycopg2.connect(
    host="localhost",
    dbname="vectordb",
    user="postgres",
    password="secret"
)

conn.autocommit = True
register_vector(conn)

cur = conn.cursor()

# Добавляем 3 новых продукта
nouveaux_produits = [
    ("chocolat", np.array([0.95, 0.05, 0.10])),
    ("gingembre", np.array([0.20, 0.80, 0.50])),
    ("litchi", np.array([0.85, 0.02, 0.95]))
]

for nom, vec in nouveaux_produits:
    cur.execute(
        """
        INSERT INTO produits (nom, caracteristiques)
        VALUES (%s, %s)
        """,
        (nom, vec)
    )

print(f"✅ {len(nouveaux_produits)} produits ajoutés")

# Vérifier le nombre total de lignes
cur.execute("SELECT COUNT(*) FROM produits;")
count = cur.fetchone()[0]

print(f"Nombre total de produits : {count}")

# Afficher tous les produits
cur.execute("""
    SELECT id, nom, caracteristiques
    FROM produits
    ORDER BY id;
""")

print("\nListe des produits :")
for id_, nom, vec in cur.fetchall():
    print(f"[{id_:2d}] {nom:12s} -> {np.round(vec, 2)}")

cur.close()
conn.close()

✅ 3 produits ajoutés
Nombre total de produits : 10

Liste des produits :
[ 1] mangue       -> [0.9  0.1  0.95]
[ 2] piment       -> [0.1  0.95 0.6 ]
[ 3] fraise       -> [0.85 0.05 0.2 ]
[ 4] curry        -> [0.2 0.9 0.7]
[ 5] vanille      -> [0.95 0.02 0.3 ]
[ 6] wasabi       -> [0.05 0.98 0.55]
[ 7] ananas       -> [0.8  0.08 0.88]
[ 8] chocolat     -> [0.95 0.05 0.1 ]
[ 9] gingembre    -> [0.2 0.8 0.5]
[10] litchi       -> [0.85 0.02 0.95]


### 2 Vecteurs & similarité cosinus 
---
<=> Косинусное расстояние

<-> Евклидово расстояние

<#> Inner Product Считает скалярное произведение

In [7]:
def distance_cosinus(a: np.ndarray, b: np.ndarray) -> float:
    """
    Calcule la distance cosinus entre deux vecteurs.
    Formule : 1 - (a · b) / (|a| × |b|)
    
    Résultat :
    - 0.0 → vecteurs identiques (même direction)
    - 1.0 → vecteurs orthogonaux (perpendiculaires)
    - 2.0 → vecteurs opposés
    """
    # Produit scalaire
    dot = np.dot(a, b)
    # Normes
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    # Similarité cosinus (entre -1 et 1)
    similarite = dot / (norm_a * norm_b)
    # Distance cosinus (entre 0 et 2)
    return 1 - similarite


In [8]:
# Nos vecteurs [sucré, épicé, exotique]
mangue   = np.array([0.9, 0.1, 0.95])
ananas   = np.array([0.8, 0.08, 0.88])
piment   = np.array([0.1, 0.95, 0.6])
vanille  = np.array([0.95, 0.02, 0.3])

print("Distances cosinus calculées manuellement :")
print(f"  mangue  ↔ ananas  : {distance_cosinus(mangue, ananas):.4f}  (proches : tous deux sucrés/exotiques)")
print(f"  mangue  ↔ piment  : {distance_cosinus(mangue, piment):.4f}  (éloignés : sucré vs épicé)")
print(f"  mangue  ↔ vanille : {distance_cosinus(mangue, vanille):.4f}  (moyens : sucrés mais exotique différent)")
print(f"  piment  ↔ vanille : {distance_cosinus(piment, vanille):.4f}  (très éloignés)")

# Illustrer l'effet de la magnitude (la distance cosinus ne change pas)
mangue_x10 = mangue * 10
print(f"\n  mangue × 10 ↔ ananas : {distance_cosinus(mangue_x10, ananas):.4f}")
print("  → La distance cosinus est insensible à la magnitude !")

# En comparaison, la distance L2 change
print(f"\nDistance L2 (Euclidienne) pour comparaison :")
print(f"  mangue  ↔ ananas  : {np.linalg.norm(mangue - ananas):.4f}")
print(f"  mangue × 10 ↔ ananas : {np.linalg.norm(mangue_x10 - ananas):.4f}")
print("  → La distance L2 est sensible à la magnitude !")

Distances cosinus calculées manuellement :
  mangue  ↔ ananas  : 0.0003  (proches : tous deux sucrés/exotiques)
  mangue  ↔ piment  : 0.4900  (éloignés : sucré vs épicé)
  mangue  ↔ vanille : 0.1268  (moyens : sucrés mais exotique différent)
  piment  ↔ vanille : 0.7384  (très éloignés)

  mangue × 10 ↔ ananas : 0.0003
  → La distance cosinus est insensible à la magnitude !

Distance L2 (Euclidienne) pour comparaison :
  mangue  ↔ ananas  : 0.1237
  mangue × 10 ↔ ananas : 11.9328
  → La distance L2 est sensible à la magnitude !


In [ ]:
# ex02_operateurs.py

conn = psycopg2.connect(host="localhost", dbname="vectordb",
                        user="postgres", password="secret")
conn.autocommit = True
register_vector(conn)
cur = conn.cursor()

# Vecteur de requête : je cherche quelque chose de "sucré et exotique"
query = np.array([1.0, 0.0, 1.0])  # sucré max, pas épicé, exotique max

print("="*55)
print("Requête : sucré=1.0, épicé=0.0, exotique=1.0")
print("="*55)

# ── Opérateur <=> : distance cosinus ──────────────────────
print("\n🔵 <=> Distance cosinus (recommandé pour textes)")
cur.execute("""
    SELECT nom,
           ROUND((1 - (caracteristiques <=> %s::vector))::numeric, 4) AS similarite,
           caracteristiques::text AS vec
    FROM produits
    ORDER BY caracteristiques <=> %s::vector
    LIMIT 4;
""", (query.tolist(), query.tolist()))
for nom, sim, vec in cur.fetchall():
    print(f"  {nom:12s}  sim={sim}  {vec}")

# ── Opérateur <-> : distance L2 (Euclidienne) ─────────────
print("\n🟢 <-> Distance L2 / Euclidienne (sensible à la magnitude)")
cur.execute("""
    SELECT nom,
           ROUND((caracteristiques <-> %s::vector)::numeric, 4) AS distance_l2
    FROM produits
    ORDER BY caracteristiques <-> %s::vector
    LIMIT 4;
""", (query.tolist(), query.tolist()))
for nom, dist in cur.fetchall():
    print(f"  {nom:12s}  dist_L2={dist}")

# ── Comparaison : même résultats ? ────────────────────────
print("\n💡 Même classement ? Les deux opérateurs donnent souvent")
print("   des résultats proches, mais pas toujours identiques.")
print("   Pour des embeddings texte normalisés, <=> est standard.")

cur.close()
conn.close()


Requête : sucré=1.0, épicé=0.0, exotique=1.0

🔵 <=> Distance cosinus (recommandé pour textes)
  litchi        sim=0.9983  [0.85,0.02,0.95]
  mangue        sim=0.9967  [0.9,0.1,0.95]
  ananas        sim=0.9966  [0.8,0.08,0.88]
  vanille       sim=0.8870  [0.95,0.02,0.3]

🟢 <-> Distance L2 / Euclidienne (sensible à la magnitude)
  mangue        dist_L2=0.1500
  litchi        dist_L2=0.1594
  ananas        dist_L2=0.2466
  vanille       dist_L2=0.7021

💡 Même classement ? Les deux opérateurs donnent souvent
   des résultats proches, mais pas toujours identiques.
   Pour des embeddings texte normalisés, <=> est standard.


In [11]:

conn = psycopg2.connect(
    host="localhost",
    dbname="vectordb",
    user="postgres",
    password="secret"
)

conn.autocommit = True
register_vector(conn)

cur = conn.cursor()

# мало сладкого, очень острое, очень экзотическое
query = np.array([0.0, 1.0, 1.0])

cur.execute("""
    SELECT nom,
           ROUND((1 - (caracteristiques <=> %s::vector))::numeric, 3) AS sim
    FROM produits
    ORDER BY caracteristiques <=> %s::vector
    LIMIT 5;
""", (query.tolist(), query.tolist()))

print("Top résultats :")
for nom, sim in cur.fetchall():
    print(f"{nom:12s} sim={sim}")

cur.close()
conn.close()

Top résultats :
curry        sim=0.977
piment       sim=0.972
wasabi       sim=0.962
gingembre    sim=0.953
ananas       sim=0.569


In [12]:
def chercher(conn, query_vec, top_k=3):
    cur = conn.cursor()

    cur.execute("""
        SELECT nom,
               ROUND(
                   (1 - (caracteristiques <=> %s::vector))::numeric,
                   4
               ) AS sim
        FROM produits
        ORDER BY caracteristiques <=> %s::vector
        LIMIT %s;
    """, (query_vec.tolist(), query_vec.tolist(), top_k))

    results = []

    for nom, sim in cur.fetchall():
        results.append({
            "nom": nom,
            "sim": float(sim)
        })

    cur.close()

    return results

In [14]:
conn = psycopg2.connect(host="localhost", dbname="vectordb",
                        user="postgres", password="secret")
conn.autocommit = True
register_vector(conn)

query = np.array([1.0, 0.0, 1.0])

resultats = chercher(conn, query)

print(resultats)
conn.close()

[{'nom': 'litchi', 'sim': 0.9983}, {'nom': 'mangue', 'sim': 0.9967}, {'nom': 'ananas', 'sim': 0.9966}]


In [27]:
def chercher_top_k(conn, query_vec, top_k=3, categorie=None):
    cur = conn.cursor()

    if np.linalg.norm(query_vec) < 1e-12:
        query_vec = np.array([0.001, 0.001, 0.001])

    if categorie is None:

        cur.execute("""
            SELECT nom,
                   ROUND(
                       (1 - (caracteristiques <=> %s::vector))::numeric,
                       4
                   ) AS sim
            FROM produits
            ORDER BY caracteristiques <=> %s::vector
            LIMIT %s;
        """, (
            query_vec.tolist(),
            query_vec.tolist(),
            top_k
        ))

    else:

        cur.execute("""
            SELECT nom,
                   ROUND(
                       (1 - (caracteristiques <=> %s::vector))::numeric,
                       4
                   ) AS sim
            FROM produits
            WHERE categorie = %s
            ORDER BY caracteristiques <=> %s::vector
            LIMIT %s;
        """, (
            query_vec.tolist(),
            categorie,
            query_vec.tolist(),
            top_k
        ))

    resultats = [
        {
            "nom": nom,
            "sim": float(sim)
        }
        for nom, sim in cur.fetchall()
    ]

    cur.close()

    return resultats

In [29]:
conn = psycopg2.connect(host="localhost", dbname="vectordb",
                        user="postgres", password="secret")
conn.autocommit = True
register_vector(conn)

query = np.array([0.0, 0.5, 0.0])

resultats = chercher_top_k(conn, query, top_k=2)

print(resultats)
conn.close()

[{'nom': 'wasabi', 'sim': 0.8712}, {'nom': 'piment', 'sim': 0.8422}]


# Index HNSW & performances
---
ANN = Approximate Nearest Neighbor

Hierarchical Navigable Small World graph

IVFFlat:

- делит пространство на кластеры
- быстрее на очень больших данных
- менее точный

HNSW:

- графовая структура
- почти всегда быстрее и точнее
- лучше для RAG

| метод   | скорость                   | размер |
| ------- | -------------------------- | ------ |
| IVFFlat | быстрее на very large data | меньше |
| HNSW    | стабильный быстрый         | больше |


In [30]:
# ex03_benchmark.py

import time

conn = psycopg2.connect(host="localhost", dbname="vectordb",
                        user="postgres", password="secret")
conn.autocommit = True
register_vector(conn)
cur = conn.cursor()

# ── Créer une table avec plus de données ────────────────────
cur.execute("""
    DROP TABLE IF EXISTS bench;
    CREATE TABLE bench (
        id        SERIAL PRIMARY KEY,
        label     TEXT,
        embedding vector(10)
    );
""")

# Générer 5000 vecteurs aléatoires (dim=10)
print("Génération de 5000 vecteurs aléatoires...")
np.random.seed(42)
n = 5000
vecs = np.random.rand(n, 10).astype(np.float32)
vecs = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)  # normalisation L2

data = [(f"item_{i}", vecs[i].tolist()) for i in range(n)]
cur.executemany(
    "INSERT INTO bench (label, embedding) VALUES (%s, %s)",
    data
)
print(f"✅ {n} lignes insérées\n")

# Vecteur de requête
query = vecs[0]  # on cherche les voisins du premier vecteur

def mesurer_query(label: str, repetitions: int = 10) -> float:
    """Exécute la requête N fois et retourne le temps moyen en ms."""
    temps = []
    for _ in range(repetitions):
        t0 = time.perf_counter()
        cur.execute("""
            SELECT label FROM bench
            ORDER BY embedding <=> %s::vector
            LIMIT 5;
        """, (query.tolist(),))
        cur.fetchall()
        temps.append((time.perf_counter() - t0) * 1000)
    moy = sum(temps) / len(temps)
    print(f"  {label:30s} : {moy:6.1f} ms  (moy sur {repetitions} requêtes)")
    return moy

# ── Sans index ───────────────────────────────────────────────
cur.execute("SET enable_indexscan = OFF;")
t_seq = mesurer_query("Sequential scan (sans index)")

# ── Création du HNSW ─────────────────────────────────────────
print("\nCréation de l'index HNSW...")
t0 = time.perf_counter()
cur.execute("""
    CREATE INDEX bench_hnsw_idx ON bench
    USING hnsw (embedding vector_cosine_ops)
    WITH (m = 16, ef_construction = 64);
""")
print(f"Index créé en {time.perf_counter()-t0:.2f}s\n")

# ── Avec index HNSW ──────────────────────────────────────────
cur.execute("SET enable_indexscan = ON;")
cur.execute("SET hnsw.ef_search = 40;")
t_hnsw = mesurer_query("HNSW (ef_search=40)")

# Effet de ef_search sur la précision/vitesse
print("\nEffet de ef_search (compromis vitesse ↔ précision) :")
for ef in [10, 20, 40, 80, 160]:
    cur.execute(f"SET hnsw.ef_search = {ef};")
    mesurer_query(f"  HNSW ef_search={ef}")

print(f"\n→ Gain de vitesse HNSW vs seq scan : ×{t_seq/t_hnsw:.1f}")
print("→ ef_search plus bas = plus rapide mais moins précis")

cur.close()
conn.close()

Génération de 5000 vecteurs aléatoires...
✅ 5000 lignes insérées

  Sequential scan (sans index)   :    1.5 ms  (moy sur 10 requêtes)

Création de l'index HNSW...
Index créé en 0.59s

  HNSW (ef_search=40)            :    0.7 ms  (moy sur 10 requêtes)

Effet de ef_search (compromis vitesse ↔ précision) :
    HNSW ef_search=10            :    0.5 ms  (moy sur 10 requêtes)
    HNSW ef_search=20            :    0.5 ms  (moy sur 10 requêtes)
    HNSW ef_search=40            :    0.6 ms  (moy sur 10 requêtes)
    HNSW ef_search=80            :    0.7 ms  (moy sur 10 requêtes)
    HNSW ef_search=160           :    0.8 ms  (moy sur 10 requêtes)

→ Gain de vitesse HNSW vs seq scan : ×2.0
→ ef_search plus bas = plus rapide mais moins précis


In [32]:
# CONNECTION ─────────────────────────────
conn = psycopg2.connect(
    host="localhost",
    dbname="vectordb",
    user="postgres",
    password="secret"
)

conn.autocommit = True
register_vector(conn)
cur = conn.cursor()

# TABLE RESET ─────────────────────────────
cur.execute("""
DROP TABLE IF EXISTS bench;
CREATE TABLE bench (
    id SERIAL PRIMARY KEY,
    label TEXT,
    embedding vector(10)
);
""")

# DATA GENERATION ─────────────────────────────
print("Generating data...")

np.random.seed(42)
n = 50000

vecs = np.random.rand(n, 10).astype(np.float32)
vecs = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)

data = [(f"item_{i}", vecs[i].tolist()) for i in range(n)]

cur.executemany(
    "INSERT INTO bench (label, embedding) VALUES (%s, %s)",
    data
)

print(f"Inserted {n} rows\n")

query = vecs[0]

# BENCH FUNCTION ─────────────────────────────
def benchmark(name, repetitions=10):
    times = []

    for _ in range(repetitions):
        t0 = time.perf_counter()

        cur.execute("""
            SELECT label
            FROM bench
            ORDER BY embedding <=> %s::vector
            LIMIT 5;
        """, (query.tolist(),))

        cur.fetchall()

        times.append((time.perf_counter() - t0) * 1000)

    avg = sum(times) / len(times)
    print(f"{name:30s} : {avg:.2f} ms")
    return avg

# 1. SEQUENTIAL SCAN  ─────────────────────────────
cur.execute("SET enable_indexscan = OFF;")
t_seq = benchmark("Sequential scan")

# 2. HNSW INDEX ─────────────────────────────
print("\nCreating HNSW index...")

cur.execute("""
    CREATE INDEX bench_hnsw_idx
    ON bench
    USING hnsw (embedding vector_cosine_ops)
    WITH (m = 16, ef_construction = 64);
""")

cur.execute("SET enable_indexscan = ON;")
cur.execute("SET hnsw.ef_search = 40;")

t_hnsw = benchmark("HNSW (ef_search=40)")

# 3. IVFFLAT INDEX ─────────────────────────────
print("\nCreating IVFFlat index...")

cur.execute("""
    CREATE INDEX bench_ivf_idx
    ON bench
    USING ivfflat (embedding vector_cosine_ops)
    WITH (lists = 50);
""")

# IMPORTANT: ivfflat settings
cur.execute("SET ivfflat.probes = 10;")

t_ivf = benchmark("IVFFlat (probes=10)")

# 4. RESULTS ─────────────────────────────
print("\nRESULTS:")
print(f"Sequential : {t_seq:.2f} ms")
print(f"HNSW       : {t_hnsw:.2f} ms")
print(f"IVFFlat    : {t_ivf:.2f} ms")

print("\nSpeedup:")
print(f"HNSW speedup vs seq : {t_seq / t_hnsw:.2f}x")
print(f"IVF speedup vs seq  : {t_seq / t_ivf:.2f}x")

# 5. EXPLAIN CHECK ─────────────────────────────
print("\nEXPLAIN (HNSW):")

cur.execute("""
    EXPLAIN (ANALYZE, BUFFERS)
    SELECT label
    FROM bench
    ORDER BY embedding <=> %s::vector
    LIMIT 5;
""", (query.tolist(),))

for row in cur.fetchall():
    print(row[0])

cur.close()
conn.close()

Generating data...
Inserted 50000 rows

Sequential scan                : 11.18 ms

Creating HNSW index...
HNSW (ef_search=40)            : 0.84 ms

Creating IVFFlat index...
IVFFlat (probes=10)            : 0.78 ms

RESULTS:
Sequential : 11.18 ms
HNSW       : 0.84 ms
IVFFlat    : 0.78 ms

Speedup:
HNSW speedup vs seq : 13.37x
IVF speedup vs seq  : 14.29x

EXPLAIN (HNSW):
Limit  (cost=110.85..112.00 rows=5 width=17) (actual time=0.357..0.362 rows=5 loops=1)
  Buffers: shared hit=683
  ->  Index Scan using bench_hnsw_idx on bench  (cost=110.85..11644.00 rows=50000 width=17) (actual time=0.356..0.360 rows=5 loops=1)
        Order By: (embedding <=> '[0.19730787,0.5008366,0.38561463,0.3153735,0.08219068,0.08217797,0.030598467,0.45630187,0.3166676,0.3730129]'::vector)
        Buffers: shared hit=683
Planning:
  Buffers: shared hit=2
Planning Time: 0.063 ms
Execution Time: 0.373 ms


##  4 — Mini-projet : moteur de recommandation de films

In [44]:
# ex04_films_setup.py

conn = psycopg2.connect(host="localhost", dbname="vectordb",
                        user="postgres", password="secret")
conn.autocommit = True
register_vector(conn)
cur = conn.cursor()

cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")

# Schéma : films avec profil vectoriel + métadonnées
cur.execute("""
    DROP TABLE IF EXISTS films;
    CREATE TABLE films (
        id      SERIAL PRIMARY KEY,
        titre   TEXT NOT NULL,
        annee   INT,
        note    NUMERIC(3,1),
        -- profil : [action, comedie, drame, romance, scifi]
        profil  vector(5)
    );
""")

# Catalogue de films avec leur profil
# Dimensions : [action, comédie, drame, romance, sci-fi]
catalogue = [
    ("Avengers: Endgame",  2019, 8.4, [0.95, 0.15, 0.35, 0.10, 0.6]),
    ("La La Land",         2016, 8.0, [0.05, 0.30, 0.55, 0.90, 0.0]),
    ("Inception",          2010, 8.8, [0.70, 0.05, 0.50, 0.20, 0.9]),
    ("Le Diable s'habille en Prada", 2006, 6.9, [0.05, 0.65, 0.45, 0.40, 0.0]),
    ("Mad Max: Fury Road",  2015, 8.1, [0.98, 0.02, 0.20, 0.05, 0.4]),
    ("Eternal Sunshine",   2004, 8.3, [0.00, 0.20, 0.75, 0.90, 0.2]),
    ("Interstellar",       2014, 8.6, [0.45, 0.05, 0.70, 0.20, 0.95]),
    ("Superbad",           2007, 7.6, [0.10, 0.92, 0.30, 0.25, 0.0]),
    ("The Dark Knight",    2008, 9.0, [0.88, 0.05, 0.65, 0.10, 0.3]),
    ("Amélie",             2001, 8.3, [0.05, 0.55, 0.60, 0.75, 0.1]),
    ("Aliens",             1986, 8.4, [0.80, 0.08, 0.40, 0.05, 0.95]),
    ("Forrest Gump",       1994, 8.8, [0.20, 0.40, 0.85, 0.60, 0.0]),
    ("John Wick",          2014, 7.4, [0.98, 0.05, 0.15, 0.05, 0.1]),
    ("The Notebook",       2004, 7.9, [0.02, 0.15, 0.60, 0.95, 0.0]),
    ("Ex Machina",         2014, 7.7, [0.15, 0.02, 0.55, 0.15, 0.95]),
]

for titre, annee, note, profil in catalogue:
    cur.execute(
        "INSERT INTO films (titre, annee, note, profil) VALUES (%s, %s, %s, %s)",
        (titre, annee, note, np.array(profil))
    )

# Créer un index HNSW pour la colonne profil
cur.execute("""
    CREATE INDEX ON films
    USING hnsw (profil vector_cosine_ops)
    WITH (m = 8, ef_construction = 40);
""")

print(f"✅ {len(catalogue)} films insérés et indexés")
print("\nDimensions du profil : [action, comédie, drame, romance, sci-fi]")

cur.close()
conn.close()


✅ 15 films insérés et indexés

Dimensions du profil : [action, comédie, drame, romance, sci-fi]


In [45]:
# ex04_recommandation.py


# ── Connexion ────────────────────────────────────────────────
conn = psycopg2.connect(host="localhost", dbname="vectordb",
                        user="postgres", password="secret")
conn.autocommit = True
register_vector(conn)
cur = conn.cursor()

# ── Fonctions utilitaires ────────────────────────────────────

def creer_profil(action=0.0, comedie=0.0, drame=0.0,
                 romance=0.0, scifi=0.0) -> np.ndarray:
    """
    Crée un vecteur de profil utilisateur.
    Chaque valeur est entre 0 (pas intéressé) et 1 (très intéressé).
    """
    return np.array([action, comedie, drame, romance, scifi], dtype=np.float32)


def recommander(profil_user: np.ndarray,
                top_k: int = 5,
                note_min: float = 0.0,
                annee_min: int = 1900) -> list[dict]:
    """
    Recommande des films similaires au profil utilisateur.
    
    Paramètres :
    - profil_user : vecteur numpy [action, comédie, drame, romance, sci-fi]
    - top_k : nombre de films à retourner
    - note_min : note IMDb minimale (filtre SQL)
    - annee_min : année de sortie minimale (filtre SQL)
    
    Retourne une liste de dicts avec titre, annee, note, similarite.
    """
    cur.execute("""
        SELECT titre,
               annee,
               note,
               ROUND((1 - (profil <=> %s::vector))::numeric, 3) AS sim
        FROM films
        WHERE note >= %s
          AND annee >= %s
        ORDER BY profil <=> %s::vector
        LIMIT %s;
    """, (profil_user.tolist(), note_min, annee_min,
          profil_user.tolist(), top_k))
    
    return [
        {"titre": row[0], "annee": row[1], "note": float(row[2]), "sim": float(row[3])}
        for row in cur.fetchall()
    ]


def films_similaires(titre_ref: str, top_k: int = 4) -> list[dict]:
    """
    Trouve les films les plus similaires à un film de référence.
    Utile pour le mode "parce que vous avez aimé X...".
    """
    # Récupérer le profil du film de référence
    cur.execute("SELECT profil FROM films WHERE titre = %s", (titre_ref,))
    row = cur.fetchone()
    if not row:
        print(f"Film '{titre_ref}' non trouvé.")
        return []
    
    profil_ref = row[0]
    
    # Trouver les similaires (en excluant le film lui-même)
    cur.execute("""
        SELECT titre,
               annee,
               ROUND((1 - (profil <=> %s::vector))::numeric, 3) AS sim
        FROM films
        WHERE titre != %s
        ORDER BY profil <=> %s::vector
        LIMIT %s;
    """, (profil_ref, titre_ref, profil_ref, top_k))
    
    return [
        {"titre": row[0], "annee": row[1], "sim": float(row[2])}
        for row in cur.fetchall()
    ]


def afficher(titre: str, resultats: list[dict]):
    """Affiche les résultats proprement."""
    print(f"\n{'='*50}")
    print(f"  {titre}")
    print(f"{'='*50}")
    for r in resultats:
        if "note" in r:
            print(f"  [{r['sim']:.3f}]  {r['titre']:35s}  ({r['annee']})  ★{r['note']}")
        else:
            print(f"  [{r['sim']:.3f}]  {r['titre']:35s}  ({r['annee']})")


# ── Scénarios de recommandation ───────────────────────────────

# Profil 1 : fan d'action/sci-fi
profil_action_scifi = creer_profil(action=0.9, scifi=0.85, drame=0.3)
afficher("Fan d'action et sci-fi", recommander(profil_action_scifi))

# Profil 2 : amateur de comédies romantiques
profil_comedie_romance = creer_profil(comedie=0.85, romance=0.80, drame=0.4)
afficher("Amateur de comédies romantiques", recommander(profil_comedie_romance))

# Profil 3 : drames récents bien notés
profil_drame = creer_profil(drame=0.9, romance=0.5)
afficher("Drames bien notés depuis 2000 (note ≥ 8.0)",
         recommander(profil_drame, note_min=8.0, annee_min=2000))

# Films similaires
afficher("Films similaires à 'Inception'", films_similaires("Inception"))
afficher("Films similaires à 'La La Land'", films_similaires("La La Land"))

cur.close()
conn.close()


  Fan d'action et sci-fi
  [0.989]  Aliens                               (1986)  ★8.4
  [0.969]  Avengers: Endgame                    (2019)  ★8.4
  [0.961]  Inception                            (2010)  ★8.8
  [0.933]  Mad Max: Fury Road                   (2015)  ★8.1
  [0.873]  Interstellar                         (2014)  ★8.6

  Amateur de comédies romantiques
  [0.961]  Le Diable s'habille en Prada         (2006)  ★6.9
  [0.953]  Amélie                               (2001)  ★8.3
  [0.889]  Superbad                             (2007)  ★7.6
  [0.882]  La La Land                           (2016)  ★8.0
  [0.830]  Forrest Gump                         (1994)  ★8.8

  Drames bien notés depuis 2000 (note ≥ 8.0)
  [0.907]  Eternal Sunshine                     (2004)  ★8.3
  [0.836]  La La Land                           (2016)  ★8.0
  [0.799]  Amélie                               (2001)  ★8.3
  [0.554]  Interstellar                         (2014)  ★8.6
  [0.541]  The Dark Knight             

In [46]:
# ex04_films_setup.py Exercice 4 **Partie A**

conn = psycopg2.connect(host="localhost", dbname="vectordb",
                        user="postgres", password="secret")
conn.autocommit = True
register_vector(conn)
cur = conn.cursor()

cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")

catalogue += [
    ("Blade Runner 2049", 2017, 8.0, [0.60, 0.02, 0.55, 0.05, 0.98]),
    ("Titanic", 1997, 7.9, [0.05, 0.10, 0.70, 0.98, 0.0]),
    ("The Hangover", 2009, 7.7, [0.05, 0.95, 0.20, 0.10, 0.0]),
    ("Matrix", 1999, 8.7, [0.85, 0.05, 0.45, 0.05, 0.95]),
    ("Her", 2013, 8.0, [0.00, 0.20, 0.60, 0.85, 0.30]),
]

def unique_film(cur):
    cur.execute("""
    DELETE FROM films
    WHERE id IN (
        SELECT id FROM (
            SELECT id,
                ROW_NUMBER() OVER (PARTITION BY titre ORDER BY id) AS rn
            FROM films
        ) t
        WHERE rn > 1
    );
""")

for titre, annee, note, profil in catalogue:
    cur.execute(
        "INSERT INTO films (titre, annee, note, profil) VALUES (%s, %s, %s, %s)",
        (titre, annee, note, np.array(profil))
    )

# Créer un index HNSW pour la colonne profil
cur.execute("""
    CREATE INDEX ON films
    USING hnsw (profil vector_cosine_ops)
    WITH (m = 8, ef_construction = 40);
""")

unique_film(cur)

print(f"✅ {len(catalogue)} films insérés et indexés")

✅ 20 films insérés et indexés


In [47]:
# ex04_interface.py
# (nécessite d'avoir exécuté ex04_films_setup.py avant)

import psycopg2
from pgvector.psycopg2 import register_vector
import numpy as np

conn = psycopg2.connect(host="localhost", dbname="vectordb",
                        user="postgres", password="secret")
conn.autocommit = True
register_vector(conn)
cur = conn.cursor()

DIMS = ["action", "comédie", "drame", "romance", "sci-fi"]

def saisir_profil() -> np.ndarray:
    """Demande à l'utilisateur de noter chaque genre de 0 à 10."""
    print("\n📝 Notez chaque genre de 0 (pas du tout) à 10 (adore) :")
    valeurs = []
    for dim in DIMS:
        while True:
            try:
                v = float(input(f"  {dim:12s} : "))
                if 0 <= v <= 10:
                    valeurs.append(v / 10.0)  # normaliser entre 0 et 1
                    break
                print("  ⚠️  Entrez une valeur entre 0 et 10")
            except ValueError:
                print("  ⚠️  Entrez un nombre")
    return np.array(valeurs, dtype=np.float32)

def recommander_interactif(profil: np.ndarray, top_k=5):
    cur.execute("""
        SELECT titre, annee, note,
               ROUND((1 - (profil <=> %s::vector))::numeric, 3) AS sim
        FROM films
        ORDER BY profil <=> %s::vector
        LIMIT %s;
    """, (profil.tolist(), profil.tolist(), top_k))
    
    print(f"\n🎬 Recommandations pour votre profil :")
    print(f"   [{', '.join(f'{v:.1f}' for v in profil)}]")
    print()
    for titre, annee, note, sim in cur.fetchall():
        barre = "█" * int(sim * 20)
        print(f"  {barre:20s}  {sim:.3f}  {titre} ({annee}) ★{note}")


if __name__ == "__main__":
    print("🎬 Moteur de recommandation de films")
    print("   Basé sur pgvector — recherche vectorielle")
    
    while True:
        print("\n" + "-"*40)
        print("1. Créer un profil et obtenir des recommandations")
        print("2. Trouver des films similaires à un titre")
        print("3. Quitter")
        
        choix = input("\nVotre choix : ").strip()
        
        if choix == "1":
            profil = saisir_profil()
            recommander_interactif(profil)
        
        elif choix == "2":
            cur.execute("SELECT titre FROM films ORDER BY titre;")
            titres = [r[0] for r in cur.fetchall()]
            print("\nFilms disponibles :")
            for i, t in enumerate(titres, 1):
                print(f"  {i:2d}. {t}")
            try:
                idx = int(input("Numéro du film : ")) - 1
                titre_ref = titres[idx]
                cur.execute("""
                    SELECT titre, annee,
                           ROUND((1 - (profil <=> (
                               SELECT profil FROM films WHERE titre = %s
                           )))::numeric, 3) AS sim
                    FROM films WHERE titre != %s
                    ORDER BY profil <=> (SELECT profil FROM films WHERE titre = %s)
                    LIMIT 4;
                """, (titre_ref, titre_ref, titre_ref))
                print(f"\nFilms similaires à '{titre_ref}' :")
                for t, a, s in cur.fetchall():
                    print(f"  [{s}]  {t} ({a})")
            except (ValueError, IndexError):
                print("Numéro invalide")
        
        elif choix == "3":
            break

    cur.close()
    conn.close()
    print("Au revoir !")

🎬 Moteur de recommandation de films
   Basé sur pgvector — recherche vectorielle

----------------------------------------
1. Créer un profil et obtenir des recommandations
2. Trouver des films similaires à un titre
3. Quitter



📝 Notez chaque genre de 0 (pas du tout) à 10 (adore) :

🎬 Recommandations pour votre profil :
   [0.9, 0.7, 0.1, 0.3, 0.8]

  █████████████████     0.885  Avengers: Endgame (2019) ★8.4
  █████████████████     0.851  Aliens (1986) ★8.4
  ████████████████      0.838  Matrix (1999) ★8.7
  ████████████████      0.829  Inception (2010) ★8.8
  ████████████████      0.812  Mad Max: Fury Road (2015) ★8.1

----------------------------------------
1. Créer un profil et obtenir des recommandations
2. Trouver des films similaires à un titre
3. Quitter
Au revoir !


In [ ]:
def trouver_contraire(conn, titre, top_k=5):
    cur = conn.cursor()

    # взять embedding исходного фильма
    cur.execute("""
        SELECT profil
        FROM films
        WHERE titre = %s;
    """, (titre,))

    row = cur.fetchone()
    if row is None:
        return []

    query_vec = row[0]

    # поиск самых НЕ похожих (DESC)
    cur.execute("""
        SELECT titre,
               ROUND((1 - (profil <=> %s::vector))::numeric, 4) AS sim
        FROM films
        WHERE titre != %s
        ORDER BY profil <=> %s::vector DESC
        LIMIT %s;
    """, (query_vec, titre, query_vec, top_k))

    result = [
        {"titre": t, "sim": float(s)}
        for t, s in cur.fetchall()
    ]

    cur.close()
    return result

In [ ]:
def recommander(conn, titre, top_k=5):
    cur = conn.cursor()

    # 1. получить embedding фильма
    cur.execute("""
        SELECT profil
        FROM films
        WHERE titre = %s;
    """, (titre,))

    query_vec = cur.fetchone()[0]

    # 2. комбинированный score
    cur.execute("""
        SELECT titre,
               note,
               ROUND(
                   (
                       0.7 * (1 - (profil <=> %s::vector))
                       + 0.3 * (note / 10.0)
                   )::numeric
               , 4) AS score_final
        FROM films
        WHERE titre != %s
        ORDER BY score_final DESC
        LIMIT %s;
    """, (query_vec, titre, top_k))

    results = [
        {"titre": t, "note": float(n), "score": float(s)}
        for t, n, s in cur.fetchall()
    ]

    cur.close()
    return results